Training Roberta Model

In [1]:
import pandas as pd
import numpy as np
import torch



In [2]:
from datasets import Dataset

c:\Users\Gerar\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [4]:
import sys
print(sys.executable)


c:\Users\Gerar\AppData\Local\Programs\Python\Python311\python.exe


In [5]:
import sys
!"c:\Users\Gerar\AppData\Local\Programs\Python\Python311\python.exe" -m pip install datasets


In [6]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

In [7]:
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

Using device: cuda


In [8]:
#loading data set
train_df = pd.read_csv("train_70.csv")
test_df  = pd.read_csv("test_30.csv")

In [9]:
#encoding labels 
le = LabelEncoder()
train_df["label_id"] = le.fit_transform(train_df["category"])
test_df["label_id"]  = le.transform(test_df["category"])

In [10]:
train_ds = Dataset.from_pandas(train_df[["text", "label_id"]])
test_ds  = Dataset.from_pandas(test_df[["text", "label_id"]])

In [11]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding = "max_length",
        truncation =True,
        max_length =128
    )

train_enc = train_ds.map(tokenize,batched = True)
test_enc = test_ds.map(tokenize, batched = True)

# Clean up
train_enc = train_enc.remove_columns(["text"])
test_enc  = test_enc.remove_columns(["text"])

train_enc = train_enc.rename_column("label_id", "labels")
test_enc  = test_enc.rename_column("label_id", "labels")

train_enc.set_format("torch")
test_enc.set_format("torch")


Map: 100%|██████████| 2455/2455 [00:00<00:00, 19152.46 examples/s]


In [12]:
num_labels = len(le.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
import sys
print(sys.executable)  # just to confirm the env

# upgrade transformers in THIS environment
!"c:\Users\Gerar\AppData\Local\Programs\Python\Python311\python.exe" -m pip install -U "transformers"


c:\Users\Gerar\AppData\Local\Programs\Python\Python311\python.exe


In [14]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   # drop to 8 if you get CUDA OOM
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    do_train=True,
    do_eval=True,
)



In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

trainer.train()

# Run evaluation explicitly after training
metrics = trainer.evaluate()
print(metrics)


Step,Training Loss
50,1.818200
100,1.275600
150,0.887600
200,0.706400
250,0.663400
300,0.606000
350,0.545700
400,0.487700
450,0.524100
500,0.477400


{'eval_loss': 0.5247024893760681, 'eval_accuracy': 0.8118126272912424, 'eval_f1_macro': 0.8967044649408551, 'eval_runtime': 8.5124, 'eval_samples_per_second': 288.404, 'eval_steps_per_second': 18.091, 'epoch': 4.0}


In [30]:
save_path = "./roberta_saved"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


('./roberta_saved\\tokenizer_config.json',
 './roberta_saved\\special_tokens_map.json',
 './roberta_saved\\vocab.json',
 './roberta_saved\\merges.txt',
 './roberta_saved\\added_tokens.json',
 './roberta_saved\\tokenizer.json')

In [29]:
sample_texts = [
    "The ground and buildings were shaking",
    "Totally awful, the trees were on fire."
]

enc = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

with torch.no_grad():
    logits = model(**enc).logits
    probs = logits.softmax(dim=-1)
    preds = probs.argmax(dim=-1).cpu().numpy()

results = le.inverse_transform(preds)

print(results)
print(probs.cpu().numpy())


['Earthquake' 'Fire/Wildfire']
[[5.9260414e-03 6.5040720e-01 1.6873481e-02 4.5392187e-03 2.7370907e-03
  8.2851656e-02 2.2477472e-01 6.4578336e-03 5.4327329e-03]
 [1.4493091e-03 9.6970005e-04 1.6326878e-03 9.7678888e-01 1.2044563e-03
  1.3650318e-02 1.6495154e-03 1.3515644e-03 1.3036003e-03]]
